<a href="https://colab.research.google.com/github/fawkes12345/LNDb-Multimodal/blob/main/notebooks/00_dataset_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LNDb v4 Dataset Audit

## Goal

Understand the structure of LNDb v4 before training any model.

Questions:

1. How many image-annotated nodules are available?
2. How many report-derived nodules are available?
3. How many matched image-report pairs are available?
4. How many image-only and report-only nodules exist?
5. Which semantic attributes are available?

In [2]:
import pandas as pd
import numpy as np

In [3]:
from google.colab import drive

drive.mount("/content/drive")

MessageError: Error: credential propagation was unsuccessful

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/LNDb-Multimodal"
)

DATA_ROOT = PROJECT_ROOT / "data"
METADATA_DIR = DATA_ROOT / "metadata"
CT_DIR = DATA_ROOT / "ct"

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

print("Project:", PROJECT_ROOT)
print("Metadata:", METADATA_DIR)
print("CT:", CT_DIR)

In [ ]:
for folder in [
    METADATA_DIR,
    CT_DIR,
    CHECKPOINT_DIR,
    OUTPUT_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Folders ready.")

In [ ]:
import requests
from pathlib import Path

RECORD_ID = "8348419"

FILES = [
    "report.csv",
    "allNods.csv",
    "chars_trainNodules.csv",
    "rad2Fleischner.csv",
    "text2Fleischner.csv",
    "help.txt",
]

host = "zenodo.org"

for filename in FILES:
    url = (
        "https://"
        + host
        + "/records/"
        + RECORD_ID
        + "/files/"
        + filename
    )

    destination = METADATA_DIR / filename

    if destination.exists():
        print(f"Already exists: {filename}")
        continue

    print(f"Downloading {filename}...")

    response = requests.get(url, stream=True)
    response.raise_for_status()

    with open(destination, "wb") as f:
        for chunk in response.iter_content(
            chunk_size=8192
        ):
            f.write(chunk)

    print(f"Saved to: {destination}")

In [ ]:
for file in METADATA_DIR.iterdir():
    print(
        file.name,
        round(file.stat().st_size / 1024, 2),
        "KB"
    )

In [ ]:
import pandas as pd

report_df = pd.read_csv(
    METADATA_DIR / "report.csv"
)

all_nodules_df = pd.read_csv(
    METADATA_DIR / "allNods.csv"
)

chars_df = pd.read_csv(
    METADATA_DIR / "chars_trainNodules.csv"
)

rad_df = pd.read_csv(
    METADATA_DIR / "rad2Fleischner.csv"
)

text_df = pd.read_csv(
    METADATA_DIR / "text2Fleischner.csv"
)

In [ ]:
display(report_df.head())

In [ ]:
print(report_df.shape)
print(all_nodules_df.shape)

print(report_df.columns.tolist())
print(all_nodules_df.columns.tolist())

display(report_df.head())
display(all_nodules_df.head())

In [ ]:
all_nodules_df["Where"].value_counts(dropna=False)

In [ ]:
all_nodules_df["Where"].unique()

In [ ]:
num_cases = all_nodules_df["LNDbID"].nunique()

print("Number of CT cases:", num_cases)

In [ ]:
findings_per_case = (
    all_nodules_df
    .groupby("LNDbID")
    .size()
)

print(findings_per_case.describe())

In [ ]:
missing = (
    all_nodules_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    * 100
)

display(missing)

In [ ]:
print(all_nodules_df["Where"].unique())

display(
    all_nodules_df["Where"]
    .value_counts(dropna=False)
)

print(
    "Number of CT cases:",
    all_nodules_df["LNDbID"].nunique()
)

display(
    all_nodules_df
    .groupby("LNDbID")
    .size()
    .describe()
)

display(
    all_nodules_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    * 100
)

## Initial Dataset Audit

The `allNods.csv` file contains:

- 1,234 finding records
- 231 unique LNDb CT IDs
- Mean findings per CT: 5.34
- Median findings per CT: 5
- Maximum findings in one CT: 19

The largest annotation groups are:

- RadAnnotation: 732
- TextReport+RadAnnotation: 377

For the first multimodal experiments, we will use a strict cohort consisting only of `TextReport+RadAnnotation` samples. Rows marked as `created: size` or `created: texture` will be analyzed separately before inclusion.

In [1]:
matched_df = all_nodules_df[
    all_nodules_df["Where"] == "TextReport+RadAnnotation"
].copy()

print("Strict matched findings:", len(matched_df))
print("Unique CT cases:", matched_df["LNDbID"].nunique())

display(matched_df.head())

NameError: name 'all_nodules_df' is not defined